# inference-mode-step — worked example 3: Missing decorator raises the leaf-in-place error

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inference-mode-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Without `@t.inference_mode()` (or a `no_grad` block), the bare in-place update `p -= lr * p.grad` on a leaf that requires grad raises a RuntimeError about in-place operations on a leaf Variable. Adding the decorator is the one-line fix.

## Worked solution

We show the failure and the fix.

1. `BrokenSGD.step` uses the bare in-place update but is NOT decorated. Calling it after a backward pass raises a RuntimeError whose message mentions a leaf Variable that requires grad being used in an in-place operation.
2. We capture that error inside a `try/except RuntimeError` and return its message so it can be inspected.
3. `FixedSGD` is identical except `step` carries `@t.inference_mode()`. The decorator disables tracking, so the same body now succeeds.
4. We trigger the broken error, print it, then run the fixed optimizer one step to confirm it updates without raising.

In [ ]:
import torch as t

t.manual_seed(2)

class BrokenSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

class FixedSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr
    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

w = t.tensor([2.0], requires_grad=True)
((w - 1) ** 2).backward()
try:
    BrokenSGD([w], 0.1).step()
    msg = ''
except RuntimeError as e:
    msg = str(e)
print('broken raised:', bool(msg))

w2 = t.tensor([2.0], requires_grad=True)
((w2 - 1) ** 2).backward()
FixedSGD([w2], 0.1).step()
print('fixed updated:', float(w2) < 2.0)